In [7]:
from sqlalchemy import create_engine
import pandas as pd
from dotenv import load_dotenv
import os

Definir os dados de acesso a base de dados

In [2]:
utilizador= "root"
password= ""
servidor="localhost"
porta = 3306
base_dados="sakila"

Criar o motor de ligação

In [3]:
engine=create_engine(f"mysql+pymysql://{utilizador}:{password}@{servidor}:{porta}/{base_dados}")

Testar a ligação

In [4]:
with engine.connect() as conexao :
    print("Ligação estabelecida com sucesso!") 

Ligação estabelecida com sucesso!


In [12]:
#Carregar as variaveis do ficheiro .env
load_dotenv()

True

In [13]:

print("DB_USER:", os.getenv("DB_USER"))
print("DB_PASSWORD:", os.getenv("DB_PASSWORD"))
print("DB_HOST:", os.getenv("DB_HOST"))
print("DB_PORT:", os.getenv("DB_PORT"))
print("DB_NAME:", os.getenv("DB_NAME"))

DB_USER: root
DB_PASSWORD: 
DB_HOST: localhost
DB_PORT: 3306
DB_NAME: sakila


In [6]:
#Ler os dados de acesso
utilizador=os.getenv("DB_USER")
password=os.getenv("DB_PASSWORD")
servidor=os.getenv("DB_HOST")
porta = int(os.getenv("DB_PORT"))
base_dados=os.getenv("DB_NAME")

engine=create_engine(f"mysql+pymysql://{utilizador}:{password}@{servidor}:{porta}/{base_dados}")
with engine.connect() as conexao :
    print("Ligação estabelecida com sucesso!") 


Ligação estabelecida com sucesso!


In [10]:
consulta="SELECT * FROM film"
with engine.connect() as conexao :
    df_filmes = pd.read_sql(consulta,conexao)
df_filmes.head(5)

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2006-02-15 05:03:42
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2006-02-15 05:03:42
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,None,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2006-02-15 05:03:42
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,1,None,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes",2006-02-15 05:03:42
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,1,None,6,2.99,130,22.99,G,Deleted Scenes,2006-02-15 05:03:42


In [11]:
consulta1="""
SELECT film_id,title,rental_rate
from film

"""
with engine.connect() as conexao :
    df_filmes1 = pd.read_sql(consulta1,conexao)
df_filmes1.head(5)

,film_id,title,rental_rate
0,1,ACADEMY DINOSAUR,0.99
1,2,ACE GOLDFINGER,4.99
2,3,ADAPTATION HOLES,2.99
3,4,AFFAIR PREJUDICE,2.99
4,5,AFRICAN EGG,2.99


In [28]:
def importar_dados(consulta):
    with engine.connect() as conexao:
        df = pd.read_sql(consulta, conexao)
    return df

In [29]:
consulta3="""
SELECT film.title,category.name AS categoria FROM film
    INNER JOIN film_category ON film.film_id=film_category.film_id
    INNER JOIN category ON film_category.category_id=category.category_id
ORDER BY film.title
"""

dfl_sql= importar_dados(consulta3)
print(dfl_sql)

                 title    categoria
0     ACADEMY DINOSAUR  Documentary
1       ACE GOLDFINGER       Horror
2     ADAPTATION HOLES  Documentary
3     AFFAIR PREJUDICE       Horror
4          AFRICAN EGG       Family
..                 ...          ...
995     YOUNG LANGUAGE  Documentary
996         YOUTH KICK        Music
997       ZHIVAGO CORE       Horror
998  ZOOLANDER FICTION     Children
999          ZORRO ARK       Comedy

[1000 rows x 2 columns]


In [30]:
df_filmes2= importar_dados("SELECT * FROM film")
df_film_category=importar_dados("SELECT * FROM film_category")
df_categorias=importar_dados("SELECT * FROM category")

df=pd.merge(df_filmes2,df_film_category,on="film_id")
df=pd.merge(df,df_categorias,on="category_id")

df = df[["title","name"]]
df = df.rename(columns={"name":"categoria"})
df


,title,categoria
0,ACADEMY DINOSAUR,Documentary
1,ACE GOLDFINGER,Horror
2,ADAPTATION HOLES,Documentary
3,AFFAIR PREJUDICE,Horror
4,AFRICAN EGG,Family
...,...,...
995,YOUNG LANGUAGE,Documentary
996,YOUTH KICK,Music
997,ZHIVAGO CORE,Horror
998,ZOOLANDER FICTION,Children


In [31]:
resultado=df.groupby("categoria")["title"].count()
resultado

categoria
Action         64
Animation      66
Children       60
Classics       57
Comedy         58
Documentary    68
Drama          62
Family         69
Foreign        73
Games          61
Horror         56
Music          51
New            63
Sci-Fi         61
Sports         74
Travel         57
Name: title, dtype: int64

In [23]:
resultado1=df.groupby("categoria")["title"].count().sort_values(ascending=False)
resultado1

categoria
Sports         74
Foreign        73
Family         69
Documentary    68
Animation      66
Action         64
New            63
Drama          62
Sci-Fi         61
Games          61
Children       60
Comedy         58
Classics       57
Travel         57
Horror         56
Music          51
Name: title, dtype: int64

In [32]:
df_uni_cat=df["categoria"].nunique()
df_uni_cat

16

In [33]:
df_uni_cat_name=df["categoria"].unique()
df_uni_cat_name

<StringArray>
['Documentary',      'Horror',      'Family',     'Foreign',      'Comedy',
      'Sports',       'Music',    'Classics',   'Animation',      'Action',
         'New',      'Sci-Fi',       'Drama',      'Travel',       'Games',
    'Children']
Length: 16, dtype: str